```markdown
## Resources

This notebook fine-tunes the **Gemma-2-2B** model on a combined dataset of synthetic and authentic academic administration documents. The following resources are used throughout this workflow:
- **Dataset Source:** Google Drive (`/content/drive/MyDrive/capstone AIUP`)
- **Pre-trained Model:** `google/gemma-2-2b-it` (retrieved from Hugging Face)
- **Fine-Tuning Framework:** Hugging Face `trl` (SFTTrainer), `peft` (LoRA), and `bitsandbytes` (4-bit quantization)
```

# Administrative Document Classification and Information Extraction

**Project Category:** Classification and Information Extraction


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
folder_path = '/content/drive/MyDrive/capstone AIUP'
print(os.listdir(folder_path))


['DATASET_CARD.md', 'README.md', 'notebook.ipynb', 'synthetic_admin_dataset-1.csv', 'authentic_admin_dataset_cleaned-v2-1.csv']


In [3]:
import os
import pandas as pd

folder_path = '/content/drive/MyDrive/capstone AIUP'
files = os.listdir(folder_path)
print("Files in folder:", files)

synthetic_file = [f for f in files if 'synthetic' in f.lower()][0]
authentic_file = [f for f in files if 'authentic' in f.lower()][0]

synthetic_path = os.path.join(folder_path, synthetic_file)
authentic_path = os.path.join(folder_path, authentic_file)

df_synthetic = pd.read_csv(synthetic_path)
df_authentic = pd.read_csv(authentic_path)

print("Datasets loaded successfully!")
print("Synthetic Dataset shape:", df_synthetic.shape)
print("Authentic Dataset shape:", df_authentic.shape)


Files in folder: ['DATASET_CARD.md', 'README.md', 'notebook.ipynb', 'synthetic_admin_dataset-1.csv', 'authentic_admin_dataset_cleaned-v2-1.csv']
Datasets loaded successfully!
Synthetic Dataset shape: (500, 8)
Authentic Dataset shape: (30, 6)


In [4]:
def tokenize_function(examples):
      return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=512)

      from datasets import Dataset
      raw_dataset = Dataset.from_pandas(df)
      tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
      print("Dataset successfully tokenized and ready for training!")


In [5]:
# 1. Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

import os
import pandas as pd
from huggingface_hub import login
import torch
from transformers import AutoTokenizer

# 2. Login to Hugging Face
login()

# 3. Load your datasets safely
folder_path = '/content/drive/MyDrive/capstone AIUP'
files = os.listdir(folder_path)
synthetic_file = [f for f in files if 'synthetic' in f][0]
authentic_file = [f for f in files if 'authentic' in f][0]

df_synthetic = pd.read_csv(os.path.join(folder_path, synthetic_file))
df_authentic = pd.read_csv(os.path.join(folder_path, authentic_file))

# Combine them into your main 'df' variable
df = pd.concat([df_synthetic, df_authentic], ignore_index=True)
df['text'] = df['text'].astype(str)
print("Datasets loaded successfully! Total rows:", len(df))

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer ready and working!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datasets loaded successfully! Total rows: 530


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Tokenizer ready and working!


In [9]:
import sys
import os

# 1. Install & Upgrade required libraries
print("Verifying dependencies...")
!pip install -q -U bitsandbytes accelerate transformers peft

# 2. Check if bitsandbytes loads cleanly; if not, restart the runtime session automatically
try:
    import bitsandbytes as bnb
    import torch
    print(f"bitsandbytes successfully loaded! Version: {bnb.__version__}")
except (ImportError, Exception) as e:
    print(f"Import failed with error: {e}. Programmatically restarting session to load new binary paths...")
    import os
    os._exit(0)

# 3. Import PEFT & Transformers
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 4. Configure 4-bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 5. Load quantized base model
print("Loading model in 4-bit precision...")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    quantization_config=quantization_config,
    device_map="auto"
)

# 6. Prepare model for PEFT training
model = prepare_model_for_kbit_training(model)

# 7. Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
print("Model loaded, quantized, and LoRA configured successfully!")

Verifying dependencies...
bitsandbytes successfully loaded! Version: 0.50.2
Loading model in 4-bit precision...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


Model loaded, quantized, and LoRA configured successfully!


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [12]:
!pip install -U bitsandbytes
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", quantization_config=quantization_config, device_map="auto")
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, peft_config)
print("Model loaded and configured successfully!")


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


Model loaded and configured successfully!


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [ ]:
!pip install -U bitsandbytes accelerate transformers


In [13]:
# # 1. Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

import os
import pandas as pd
from huggingface_hub import login
import torch
from transformers import AutoTokenizer

# 2. Login to Hugging Face
login()

# 3. Load your datasets safely
folder_path = '/content/drive/MyDrive/capstone AIUP'
files = os.listdir(folder_path)
synthetic_file = [f for f in files if 'synthetic' in f][0]
authentic_file = [f for f in files if 'authentic' in f][0]

df_synthetic = pd.read_csv(os.path.join(folder_path, synthetic_file))
df_authentic = pd.read_csv(os.path.join(folder_path, authentic_file))

# Combine them into your main 'df' variable
df = pd.concat([df_synthetic, df_authentic], ignore_index=True)
df['text'] = df['text'].astype(str)
print("Datasets loaded successfully! Total rows:", len(df))

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer ready and working!")


Datasets loaded successfully! Total rows: 530
Tokenizer ready and working!


In [14]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "google/gemma-2-2b-it"

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Tokenizer loaded successfully!


```markdown
## Model Fine-Tuning with Supervised Fine-Tuning (SFT)
This section configures and executes the training loop for fine-tuning our quantized **Gemma-2-2B** model using Low-Rank Adaptation (LoRA) and the **SFTTrainer** from the Hugging Face `trl` library.
```

In [17]:
# Install 'trl' library if it's not present
!pip install -q trl datasets

import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Convert pandas DataFrame to a Hugging Face Dataset
train_dataset = Dataset.from_pandas(df)

# Configure training arguments using SFTConfig instead of standard TrainingArguments
training_args = SFTConfig(
    output_dir="./gemma-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    max_steps=50,
    bf16=True, # Changed from fp16=True to match bfloat16 compute datatype
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
    max_length=512
)

# Instantiate the SFT Trainer using processing_class instead of tokenizer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_args
)

print("Starting training loop...")
trainer.train()
print("Training completed successfully!")

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Adding EOS to train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Starting training loop...


Step,Training Loss
10,1.670440
20,0.736051
30,0.441033
40,0.367620
50,0.347748


Training completed successfully!


```markdown
## Save Fine-Tuned Adapter to Google Drive
This step saves the adapter configuration and weights (LoRA parameters) directly to your Google Drive folder so they are permanently stored and can be reused or deployed later.
```

In [18]:
import os

# Define export path in Google Drive
export_dir = '/content/drive/MyDrive/capstone AIUP/gemma-finetuned-adapter'

# Create directory if it doesn't exist
os.makedirs(export_dir, exist_ok=True)

# Save the fine-tuned model adapter and tokenizer
trainer.model.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)

print(f"Success! Your fine-tuned LoRA adapter and tokenizer are saved to: {export_dir}")

Success! Your fine-tuned LoRA adapter and tokenizer are saved to: /content/drive/MyDrive/capstone AIUP/gemma-finetuned-adapter


```markdown
## Model Evaluation & Inference
This step loads the newly saved LoRA adapter from Google Drive onto the base quantized Gemma model to perform test generation on administrative prompts.
```

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "google/gemma-2-2b-it"
adapter_dir = "/content/drive/MyDrive/capstone AIUP/gemma-finetuned-adapter"

# Configure 4-bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load Tokenizer and Base Model
print("Loading base model and tokenizer...")
eval_tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
eval_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# Load Fine-Tuned Adapter
print("Loading fine-tuned LoRA adapter...")
eval_model = PeftModel.from_pretrained(eval_base_model, adapter_dir)
eval_model.eval()

# Run a test prompt representing academic administration
test_prompt = "Write an official memorandum regarding the scheduling of the upcoming Academic Board meeting."

# Simple generation format
inputs = eval_tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = eval_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

print("\n--- Generated Administrative Document Output ---")
print(eval_tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading base model and tokenizer...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loading fine-tuned LoRA adapter...


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(



--- Generated Administrative Document Output ---
Write an official memorandum regarding the scheduling of the upcoming Academic Board meeting. Forward the memorandum through the appropriate departmental channel. Deadline: 2026-11-26. Please direct any procedural enquiries to Academic Planning Unit and quote the reference number in all correspondence.


```markdown
## Document Classification Pipeline
Using our fine-tuned Gemma-2-2B model, we can classify administrative prompts and texts into their corresponding `document_format` and `administrative_purpose` based on the patterns learned from our dataset.
```

In [33]:
def classify_document(text_prompt, model, tokenizer):
    """
    Classify a given text prompt or document body to determine its document format
    and administrative purpose using advanced line-by-line parsing compatible with
    both newline and punctuation separation.
    """
    # Prompt the model to complete the metadata header for the given text
    structured_prompt = f"""Analyze the document below and fill in the classification metadata.

Text: {text_prompt}

Document Format:"""

    inputs = tokenizer(structured_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False
        )

    # Extract newly generated tokens
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_part = result[len(structured_prompt):].strip()

    # Prepend "Document Format: " back for consistent parsing
    full_response = "Document Format: " + generated_part

    # Parse fields dynamically based on both newlines and punctuation
    parsed_results = {"document_format": "Unknown", "administrative_purpose": "Unknown"}

    # Split by newline first, then by period to handle both layouts safely
    raw_lines = []
    for part in full_response.split("\n"):
        raw_lines.extend(part.split("."))

    for line in raw_lines:
        line = line.strip()
        if not line:
            continue
        if "document format:" in line.lower():
            parsed_results["document_format"] = line.lower().split("document format:", 1)[1].strip().title()
        elif "administrative purpose:" in line.lower() or "purpose:" in line.lower():
            # Attempt to split on either long or short form key
            key = "administrative purpose:" if "administrative purpose:" in line.lower() else "purpose:"
            parsed_results["administrative_purpose"] = line.lower().split(key, 1)[1].strip().title()
        elif "subject:" in line.lower() and parsed_results["administrative_purpose"] == "Unknown":
            parsed_results["administrative_purpose"] = line.lower().split("subject:", 1)[1].strip().title()

    return parsed_results

# --- Demo of the Updated Parser ---
test_document = "All faculty members are requested to attend the sensitization program on Sustainable Development Goals (SDGs) in the main auditorium this Thursday at 10:00 AM."

print("Input Text:", test_document)
print("\n--- Extracted Classification ---")
classification = classify_document(test_document, eval_model, eval_tokenizer)
print(f"Document Format: {classification['document_format']}")
print(f"Administrative Purpose: {classification['administrative_purpose']}")

Input Text: All faculty members are requested to attend the sensitization program on Sustainable Development Goals (SDGs) in the main auditorium this Thursday at 10:00 AM.

--- Extracted Classification ---
Document Format: Announcement
Administrative Purpose: Sensitization Program


```markdown
## Classification Performance Evaluation
We will now evaluate our classification model over a batch of real administrative documents from `df_authentic` to check our accuracy for both `document_format` and `administrative_purpose` fields.
```

In [26]:
import pandas as pd

# df_authentic has 'content' as its document text body column
text_col = 'content'

# Filter df_authentic to include rows where required fields are present
eval_subset = df_authentic.dropna(subset=[text_col, 'document_format', 'administrative_purpose']).copy()

# Take a validation sample of 10-15 documents
eval_sample = eval_subset.sample(n=min(15, len(eval_subset)), random_state=42)

correct_format = 0
correct_purpose = 0
total_samples = len(eval_sample)

print(f"Evaluating classification accuracy on {total_samples} authentic samples using actual text body from '{text_col}'...\n")

for idx, row in eval_sample.iterrows():
    raw_text = row[text_col]
    true_format = str(row['document_format']).strip().lower()
    true_purpose = str(row['administrative_purpose']).strip().lower()

    # Generate predictions using our classifier pipeline
    pred = classify_document(raw_text, eval_model, eval_tokenizer)
    pred_format = pred['document_format'].strip().lower()
    pred_purpose = pred['administrative_purpose'].strip().lower()

    # Use substring matching to account for minor variation in text generation outputs
    fmt_match = true_format in pred_format or pred_format in true_format or any(w in pred_format for w in true_format.split())
    purp_match = true_purpose in pred_purpose or pred_purpose in true_purpose or any(w in pred_purpose for w in true_purpose.split() if len(w) > 3)

    if fmt_match:
        correct_format += 1
    if purp_match:
        correct_purpose += 1

    print(f"Document Text snippet: '{str(raw_text)[:80]}...'")
    print(f"  -> True Format : {true_format.title()} | Predicted: {pred['document_format']}")
    print(f"  -> True Purpose: {true_purpose.title()} | Predicted: {pred['administrative_purpose']}")
    print(f"  -> Format Match: {fmt_match} | Purpose Match: {purp_match}\n")

print("--------------------------------------------------")
print(f"Document Format Accuracy      : {correct_format / total_samples * 100:.2f}%")
print(f"Administrative Purpose Accuracy: {correct_purpose / total_samples * 100:.2f}%")

Evaluating classification accuracy on 15 authentic samples using actual text body from 'content'...

Document Text snippet: 'University ACADEMICS RANK AMONG TWO PERCENT(2%) IN 2026 STANFORD-ELSEVIER WORLD’...'
  -> True Format : Notice | Predicted: Press Release
  -> True Purpose: General Notice | Predicted: Unknown
  -> Format Match: False | Purpose Match: False

Document Text snippet: 'INVITATION TO A CAMPUS-WIDE FACULTY SDGs SENSITIZATION OUTREACH
The Centre for S...'
  -> True Format : Invitation | Predicted: Letter
  -> True Purpose: Student / Staff Programme | Predicted: Invitation To A Campus-Wide Faculty Sdgs Sensitization Outreach
  -> Format Match: False | Purpose Match: False

Document Text snippet: 'MASTER MOLECULAR BIOLOGY TECHNIQUES — HANDS-ON
Are you a student, researcher, la...'
  -> True Format : Notice | Predicted: Training Schedule
  -> True Purpose: General Notice | Predicted: Unknown
  -> Format Match: False | Purpose Match: False

Document Text snippet: '5th Facul

```markdown
## Investigation: Analyzing Training Text Structure
Let's print a few samples from our processed training texts in `df` to see how metadata fields (like `document_format` and `administrative_purpose`) were formatted during fine-tuning.
```

In [27]:
# Inspect the first few training text blocks
for i in range(3):
    print(f"--- Training Sample #{i+1} ---")
    print(df['text'].iloc[i])
    print("\n" + "="*50 + "\n")

--- Training Sample #1 ---
Reference: ADM/2026/0001. From: Department of Computer Science. To: The Vice-Chancellor. Subject: Request for approval. This communication concerns the following matter: the unit requests authorization to conduct a departmental capacity-building workshop. Attend the scheduled briefing session and sign the attendance register. Deadline: 2026-11-29. Please direct any procedural enquiries to Department of Computer Science and quote the reference number in all correspondence.


--- Training Sample #2 ---
Reference: ADM/2026/0002. From: Office of the Registrar. To: Relevant Unit Heads. Subject: Appointment. This communication concerns the following matter: the recipient is appointed to serve on the designated university committee. Acknowledge receipt and proceed in accordance with the attached guidance. Deadline: 2026-12-08. Please direct any procedural enquiries to Office of the Registrar and quote the reference number in all correspondence.


--- Training Sample

```markdown
## Create Instruction-Based Dataset Version
To align training with structured instruction-following, we will reformulate the `'text'` column in our dataset. We will wrap the raw document body into a prompt asking the model to extract and output `Document Format` and `Administrative Purpose` explicitly. This allows the model to learn the association directly during fine-tuning.
```

In [28]:
import pandas as pd
import os

# Reload original clean datasets
folder_path = '/content/drive/MyDrive/capstone AIUP'
synthetic_file = [f for f in os.listdir(folder_path) if 'synthetic' in f.lower()][0]
authentic_file = [f for f in os.listdir(folder_path) if 'authentic' in f.lower()][0]

df_syn = pd.read_csv(os.path.join(folder_path, synthetic_file))
df_auth = pd.read_csv(os.path.join(folder_path, authentic_file))

# Standardize text columns (synthetic uses 'text', authentic uses 'content' for raw body)
df_syn['raw_content'] = df_syn['text']
df_auth['raw_content'] = df_auth['content']

# Combine datasets safely
df_combined = pd.concat([
    df_syn[['raw_content', 'document_format', 'administrative_purpose']],
    df_auth[['raw_content', 'document_format', 'administrative_purpose']]
], ignore_index=True)

# Define the Instruction Prompt Template
def create_instruction_format(row):
    document_text = str(row['raw_content']).strip()
    doc_format = str(row['document_format']).strip()
    purpose = str(row['administrative_purpose']).strip()

    prompt = f"""Analyze the document below and fill in the classification metadata.

Text: {document_text}

Document Format: {doc_format}
Administrative Purpose: {purpose}"""
    return prompt

# Apply the template to create a new 'text' column
df_combined['text'] = df_combined.apply(create_instruction_format, axis=1)

# Save the new dataset version to Google Drive
output_filename = 'instruction_formatted_admin_dataset.csv'
output_filepath = os.path.join(folder_path, output_filename)
df_combined.to_csv(output_filepath, index=False)

print(f"New dataset version successfully saved! Length: {len(df_combined)} rows")
print(f"File location: {output_filepath}")
print("\n--- Sample Formatted Instruction --- ")
print(df_combined['text'].iloc[0])

New dataset version successfully saved! Length: 530 rows
File location: /content/drive/MyDrive/capstone AIUP/instruction_formatted_admin_dataset.csv

--- Sample Formatted Instruction --- 
Analyze the document below and fill in the classification metadata.

Text: Reference: ADM/2026/0001. From: Department of Computer Science. To: The Vice-Chancellor. Subject: Request for approval. This communication concerns the following matter: the unit requests authorization to conduct a departmental capacity-building workshop. Attend the scheduled briefing session and sign the attendance register. Deadline: 2026-11-29. Please direct any procedural enquiries to Department of Computer Science and quote the reference number in all correspondence.

Document Format: Memorandum (memo)
Administrative Purpose: Request for approval


```markdown
## SFT Fine-Tuning with Instruction-Formatted Dataset
Now, we load the newly created `instruction_formatted_admin_dataset.csv` and fine-tune our quantized Gemma-2-2B model. This ensures it learns how to directly output structured classifications based on our prompting schema.
```

In [29]:
import pandas as pd
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Load the instruction-formatted dataset
instruction_dataset_path = "/content/drive/MyDrive/capstone AIUP/instruction_formatted_admin_dataset.csv"
df_instruction = pd.read_csv(instruction_dataset_path)

# Convert to Hugging Face Dataset
train_instruction_dataset = Dataset.from_pandas(df_instruction)

# Configure training parameters
training_args_inst = SFTConfig(
    output_dir="./gemma-instruction-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    max_steps=50,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
    max_length=512
)

# Initialize Trainer
trainer_inst = SFTTrainer(
    model=model,
    train_dataset=train_instruction_dataset,
    processing_class=tokenizer,
    args=training_args_inst
)

print("Starting fine-tuning with the instruction-formatted dataset...")
trainer_inst.train()
print("Training completed successfully!")

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Adding EOS to train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/530 [00:00<?, ? examples/s]

Starting fine-tuning with the instruction-formatted dataset...


Step,Training Loss
10,0.706933
20,0.582396
30,0.250029
40,0.294425
50,0.418540


Training completed successfully!


```markdown
## Save New Instruction Adapter
We save this instruction-tuned model adapter to a new path on Google Drive so we can test its performance.
```

In [30]:
import os

new_export_dir = '/content/drive/MyDrive/capstone AIUP/gemma-instruction-adapter'
os.makedirs(new_export_dir, exist_ok=True)

trainer_inst.model.save_pretrained(new_export_dir)
tokenizer.save_pretrained(new_export_dir)

print(f"Success! Your new instruction-aligned LoRA adapter is saved to: {new_export_dir}")

Success! Your new instruction-aligned LoRA adapter is saved to: /content/drive/MyDrive/capstone AIUP/gemma-instruction-adapter


## Re-Evaluating Classification Accuracy with the Instruction-Fine-Tuned Adapter
Now we load our newly trained instruction adapter, run it over our validation sample from `df_authentic`, and compare the predictions to verify our updated classification accuracy.

In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import pandas as pd

base_model_id = "google/gemma-2-2b-it"
instruction_adapter_dir = "/content/drive/MyDrive/capstone AIUP/gemma-instruction-adapter"

# Configure 4-bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load Tokenizer and Base Model
print("Loading base model and tokenizer...")
inst_tokenizer = AutoTokenizer.from_pretrained(instruction_adapter_dir)
inst_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# Load the new instruction-tuned LoRA Adapter
print("Loading instruction-tuned LoRA adapter...")
inst_model = PeftModel.from_pretrained(inst_base_model, instruction_adapter_dir)
inst_model.eval()

# Re-run evaluation loop
correct_format = 0
correct_purpose = 0
total_samples = len(eval_sample)

print(f"\nEvaluating classification accuracy on {total_samples} authentic samples...\n")

for idx, row in eval_sample.iterrows():
    raw_text = row['content']
    true_format = str(row['document_format']).strip().lower()
    true_purpose = str(row['administrative_purpose']).strip().lower()

    # Generate predictions using our structured classify_document function
    pred = classify_document(raw_text, inst_model, inst_tokenizer)
    pred_format = pred['document_format'].strip().lower()
    pred_purpose = pred['administrative_purpose'].strip().lower()

    # Evaluate matches (using substring and basic word matching to account for naming variations)
    fmt_match = true_format in pred_format or pred_format in true_format or any(w in pred_format for w in true_format.split() if len(w) > 3)
    purp_match = true_purpose in pred_purpose or pred_purpose in true_purpose or any(w in pred_purpose for w in true_purpose.split() if len(w) > 3)

    if fmt_match:
        correct_format += 1
    if purp_match:
        correct_purpose += 1

    print(f"Document Text snippet: '{str(raw_text)[:80]}...'")
    print(f"  -> True Format : {true_format.title()} | Predicted: {pred['document_format']}")
    print(f"  -> True Purpose: {true_purpose.title()} | Predicted: {pred['administrative_purpose']}")
    print(f"  -> Format Match: {fmt_match} | Purpose Match: {purp_match}\n")

print("--------------------------------------------------")
print(f"Document Format Accuracy      : {correct_format / total_samples * 100:.2f}%")
print(f"Administrative Purpose Accuracy: {correct_purpose / total_samples * 100:.2f}%")

Loading base model and tokenizer...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loading instruction-tuned LoRA adapter...


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(



Evaluating classification accuracy on 15 authentic samples...

Document Text snippet: 'University ACADEMICS RANK AMONG TWO PERCENT(2%) IN 2026 STANFORD-ELSEVIER WORLD’...'
  -> True Format : Notice | Predicted: Letter
Administrative Purpose: Notice
  -> True Purpose: General Notice | Predicted: Unknown
  -> Format Match: True | Purpose Match: False

Document Text snippet: 'INVITATION TO A CAMPUS-WIDE FACULTY SDGs SENSITIZATION OUTREACH
The Centre for S...'
  -> True Format : Invitation | Predicted: Letter
Administrative Purpose: Invitation To A Campus-Wide Faculty Sdgs Sensitization Outreach
  -> True Purpose: Student / Staff Programme | Predicted: Unknown
  -> Format Match: True | Purpose Match: False

Document Text snippet: 'MASTER MOLECULAR BIOLOGY TECHNIQUES — HANDS-ON
Are you a student, researcher, la...'
  -> True Format : Notice | Predicted: Letter
Administrative Purpose: Training
Administrative Purpose: Training
  -> True Purpose: General Notice | Predicted: Unknown
  -> Forma

```markdown
## Re-Evaluating Classification Accuracy with the Instruction-Fine-Tuned Adapter
Now we load our newly trained instruction adapter, run it over our validation sample from `df_authentic`, and compare the predictions to verify our updated classification accuracy.
```

In [32]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import pandas as pd

base_model_id = "google/gemma-2-2b-it"
instruction_adapter_dir = "/content/drive/MyDrive/capstone AIUP/gemma-instruction-adapter"

# Configure 4-bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load Tokenizer and Base Model
print("Loading base model and tokenizer...")
inst_tokenizer = AutoTokenizer.from_pretrained(instruction_adapter_dir)
inst_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# Load the new instruction-tuned LoRA Adapter
print("Loading instruction-tuned LoRA adapter...")
inst_model = PeftModel.from_pretrained(inst_base_model, instruction_adapter_dir)
inst_model.eval()

# Re-run evaluation loop
correct_format = 0
correct_purpose = 0
total_samples = len(eval_sample)

print(f"\nEvaluating classification accuracy on {total_samples} authentic samples...\n")

for idx, row in eval_sample.iterrows():
    raw_text = row['content']
    true_format = str(row['document_format']).strip().lower()
    true_purpose = str(row['administrative_purpose']).strip().lower()

    # Generate predictions using our structured classify_document function
    pred = classify_document(raw_text, inst_model, inst_tokenizer)
    pred_format = pred['document_format'].strip().lower()
    pred_purpose = pred['administrative_purpose'].strip().lower()

    # Evaluate matches (using substring and basic word matching to account for naming variations)
    fmt_match = true_format in pred_format or pred_format in true_format or any(w in pred_format for w in true_format.split() if len(w) > 3)
    purp_match = true_purpose in pred_purpose or pred_purpose in true_purpose or any(w in pred_purpose for w in true_purpose.split() if len(w) > 3)

    if fmt_match:
        correct_format += 1
    if purp_match:
        correct_purpose += 1

    print(f"Document Text snippet: '{str(raw_text)[:80]}...'")
    print(f"  -> True Format : {true_format.title()} | Predicted: {pred['document_format']}")
    print(f"  -> True Purpose: {true_purpose.title()} | Predicted: {pred['administrative_purpose']}")
    print(f"  -> Format Match: {fmt_match} | Purpose Match: {purp_match}\n")

print("--------------------------------------------------")
print(f"Document Format Accuracy      : {correct_format / total_samples * 100:.2f}%")
print(f"Administrative Purpose Accuracy: {correct_purpose / total_samples * 100:.2f}%")

Loading base model and tokenizer...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loading instruction-tuned LoRA adapter...

Evaluating classification accuracy on 15 authentic samples...

Document Text snippet: 'University ACADEMICS RANK AMONG TWO PERCENT(2%) IN 2026 STANFORD-ELSEVIER WORLD’...'
  -> True Format : Notice | Predicted: Letter
Administrative Purpose: Notice
  -> True Purpose: General Notice | Predicted: Unknown
  -> Format Match: True | Purpose Match: False

Document Text snippet: 'INVITATION TO A CAMPUS-WIDE FACULTY SDGs SENSITIZATION OUTREACH
The Centre for S...'
  -> True Format : Invitation | Predicted: Letter
Administrative Purpose: Invitation To A Campus-Wide Faculty Sdgs Sensitization Outreach
  -> True Purpose: Student / Staff Programme | Predicted: Unknown
  -> Format Match: True | Purpose Match: False

Document Text snippet: 'MASTER MOLECULAR BIOLOGY TECHNIQUES — HANDS-ON
Are you a student, researcher, la...'
  -> True Format : Notice | Predicted: Letter
Administrative Purpose: Training
Administrative Purpose: Training
  -> True Purpose: Gene

## Re-Run Accuracy Evaluation with New Multi-Line Parser
Now we re-run our evaluation loop using the newly updated `classify_document` function to see our final improved classification metrics on authentic data.

In [34]:
correct_format = 0
correct_purpose = 0
total_samples = len(eval_sample)

print(f"Evaluating classification accuracy on {total_samples} authentic samples...\n")

for idx, row in eval_sample.iterrows():
    raw_text = row['content']
    true_format = str(row['document_format']).strip().lower()
    true_purpose = str(row['administrative_purpose']).strip().lower()

    # Use updated multi-line parsing
    pred = classify_document(raw_text, inst_model, inst_tokenizer)
    pred_format = pred['document_format'].strip().lower()
    pred_purpose = pred['administrative_purpose'].strip().lower()

    # Evaluate matches (using substring and basic word matching to account for naming variations)
    fmt_match = true_format in pred_format or pred_format in true_format or any(w in pred_format for w in true_format.split() if len(w) > 3)
    purp_match = true_purpose in pred_purpose or pred_purpose in true_purpose or any(w in pred_purpose for w in true_purpose.split() if len(w) > 3)

    if fmt_match:
        correct_format += 1
    if purp_match:
        correct_purpose += 1

    print(f"Document Text: '{str(raw_text)[:60]}...'")
    print(f"  -> True Format : {true_format.title()} | Predicted: {pred['document_format']}")
    print(f"  -> True Purpose: {true_purpose.title()} | Predicted: {pred['administrative_purpose']}")
    print(f"  -> Format Match: {fmt_match} | Purpose Match: {purp_match}\n")

print("--------------------------------------------------")
print(f"Final Document Format Accuracy      : {correct_format / total_samples * 100:.2f}%")
print(f"Final Administrative Purpose Accuracy: {correct_purpose / total_samples * 100:.2f}%")

Evaluating classification accuracy on 15 authentic samples...

Document Text: 'University ACADEMICS RANK AMONG TWO PERCENT(2%) IN 2026 STAN...'
  -> True Format : Notice | Predicted: Letter
  -> True Purpose: General Notice | Predicted: Notice
  -> Format Match: False | Purpose Match: True

Document Text: 'INVITATION TO A CAMPUS-WIDE FACULTY SDGs SENSITIZATION OUTRE...'
  -> True Format : Invitation | Predicted: Letter
  -> True Purpose: Student / Staff Programme | Predicted: Invitation To A Campus-Wide Faculty Sdgs Sensitization Outreach
  -> Format Match: False | Purpose Match: False

Document Text: 'MASTER MOLECULAR BIOLOGY TECHNIQUES — HANDS-ON
Are you a stu...'
  -> True Format : Notice | Predicted: Letter
  -> True Purpose: General Notice | Predicted: Training
  -> Format Match: False | Purpose Match: False

Document Text: '5th Faculty of Life Sciences Annual Scientific Conference 20...'
  -> True Format : Notice | Predicted: Article
  -> True Purpose: Training / Event / Conferen

## Export Model and Datasets to GitHub
This section authenticates and pushes our instruction-aligned LoRA adapter and dataset files to your public repository: `https://github.com/edla4eva/capstone.git`.

In [43]:
import os
from google.colab import userdata

# --- CONFIGURATION ---
# Fetching your Personal Access Token from Colab Secrets.
# Please ensure you have added "GITHUB_TOKEN" in your Colab secrets with your real GitHub PAT.
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("Successfully retrieved GITHUB_TOKEN from Secrets!")
except Exception as e:
    print("Could not retrieve GITHUB_TOKEN from Secrets. Please add it to your Colab Secrets.")
    GITHUB_TOKEN = None

REPO_URL = "github.com/edla4eva/capstone.git"
CLONE_DIR = "/content/drive/MyDrive/capstone_repo"

# Configure Git identity
!git config --global user.email "edla4eva@yahoo.com"
!git config --global user.name "edla4eva"

# Clean up any existing directory and clone
if os.path.exists(CLONE_DIR):
    !rm -rf {CLONE_DIR}

print("Cloning repository...")
if GITHUB_TOKEN:
    # Authenticated clone allows pushing later without interactive prompts
    !git clone https://{GITHUB_TOKEN}@{REPO_URL} {CLONE_DIR}
else:
    print("Warning: Attempting public pull without write permission (Push will fail without PAT).")
    !git clone https://{REPO_URL} {CLONE_DIR}

Successfully retrieved GITHUB_TOKEN from Secrets!
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning repository...
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/drive/MyDrive/capstone_repo'...
fatal: Unable to read current working directory: No such file or directory


In [44]:

import os
import shutil
import subprocess
import base64
from pathlib import Path
from google.colab import userdata, drive

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

REPO_OWNER = "edla4eva"
REPO_NAME = "capstone"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

DRIVE_ROOT = Path("/content/drive/MyDrive")
CLONE_DIR = DRIVE_ROOT / "capstone_repo"

# --------------------------------------------------
# 1. ENSURE GOOGLE DRIVE IS MOUNTED
# --------------------------------------------------

drive.mount("/content/drive")

# IMPORTANT:
# Move outside the repository before deleting it.
os.chdir("/content")

# --------------------------------------------------
# 2. RETRIEVE GITHUB TOKEN
# --------------------------------------------------

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

    if not GITHUB_TOKEN:
        raise ValueError("GITHUB_TOKEN is empty")

    print("Successfully retrieved GITHUB_TOKEN!")

except Exception as e:
    raise RuntimeError(
        "Could not retrieve GITHUB_TOKEN. "
        "Check your Colab Secrets configuration."
    ) from e

# --------------------------------------------------
# 3. CONFIGURE GIT IDENTITY
# --------------------------------------------------

subprocess.run(
    ["git", "config", "--global", "user.email",
     "edla4eva@yahoo.com"],
    check=True
)

subprocess.run(
    ["git", "config", "--global", "user.name",
     "edla4eva"],
    check=True
)

# --------------------------------------------------
# 4. REMOVE EXISTING REPOSITORY
# --------------------------------------------------

if CLONE_DIR.exists():
    print(f"Removing existing repository: {CLONE_DIR}")
    shutil.rmtree(CLONE_DIR)

# --------------------------------------------------
# 5. AUTHENTICATE GIT
# --------------------------------------------------

# Use HTTP Basic authentication.
# The token is not embedded in the repository URL.

credentials = f"{REPO_OWNER}:{GITHUB_TOKEN}"

encoded_credentials = base64.b64encode(
    credentials.encode("utf-8")
).decode("ascii")

git_env = os.environ.copy()

git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraheader"
git_env["GIT_CONFIG_VALUE_0"] = (
    f"AUTHORIZATION: basic {encoded_credentials}"
)

# --------------------------------------------------
# 6. CLONE REPOSITORY
# --------------------------------------------------

print("Cloning repository...")

subprocess.run(
    [
        "git",
        "clone",
        REPO_URL,
        str(CLONE_DIR)
    ],
    env=git_env,
    cwd="/content",
    check=True
)

print("Repository cloned successfully!")

# --------------------------------------------------
# 7. CHANGE TO REPOSITORY DIRECTORY
# --------------------------------------------------

os.chdir(CLONE_DIR)

print(f"Current working directory: {os.getcwd()}")
print("Repository contents:")

for item in os.listdir("."):
    print(item)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully retrieved GITHUB_TOKEN!
Cloning repository...
Repository cloned successfully!
Current working directory: /content/drive/MyDrive/capstone_repo
Repository contents:
.git
DATASET_CARD.md
Readme.MD
notebook.ipynb


In [45]:

import os
import shutil
from pathlib import Path

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

# Source files in Google Drive
SOURCE_ADAPTER_DIR = Path(
    "/content/drive/MyDrive/capstone AIUP/gemma-instruction-adapter"
)

SOURCE_DATASET = Path(
    "/content/drive/MyDrive/capstone AIUP/instruction_formatted_admin_dataset.csv"
)

# Destination inside the cloned GitHub repository
REPO_DIR = Path(CLONE_DIR)

TARGET_ADAPTER_DIR = REPO_DIR / "gemma-instruction-adapter"
TARGET_DATASET = REPO_DIR / "instruction_formatted_admin_dataset.csv"

# --------------------------------------------------
# 1. VALIDATE SOURCE FILES
# --------------------------------------------------

if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Repository directory does not exist: {REPO_DIR}"
    )

if not SOURCE_ADAPTER_DIR.is_dir():
    raise FileNotFoundError(
        f"Adapter directory not found: {SOURCE_ADAPTER_DIR}"
    )

if not SOURCE_DATASET.is_file():
    raise FileNotFoundError(
        f"Dataset file not found: {SOURCE_DATASET}"
    )

# --------------------------------------------------
# 2. COPY FORMATTED DATASET
# --------------------------------------------------

print("Copying formatted dataset...")

shutil.copy2(
    SOURCE_DATASET,
    TARGET_DATASET
)

print(f"Dataset copied: {TARGET_DATASET}")

# --------------------------------------------------
# 3. COPY INSTRUCTION ADAPTER
# --------------------------------------------------

print("Copying adapter files...")

# Remove previous adapter directory if it exists
if TARGET_ADAPTER_DIR.exists():
    shutil.rmtree(TARGET_ADAPTER_DIR)

# Copy adapter directory and its contents
shutil.copytree(
    SOURCE_ADAPTER_DIR,
    TARGET_ADAPTER_DIR
)

print(f"Adapter copied: {TARGET_ADAPTER_DIR}")

# --------------------------------------------------
# 4. VERIFY COPIED FILES
# --------------------------------------------------

adapter_files = [
    p for p in TARGET_ADAPTER_DIR.rglob("*")
    if p.is_file()
]

print("\nCopy verification:")
print(f"Dataset exists: {TARGET_DATASET.is_file()}")
print(f"Adapter files copied: {len(adapter_files)}")

for file in adapter_files:
    print(f"  - {file.relative_to(REPO_DIR)}")

print("\nFiles successfully staged in the local repository!")


Copying formatted dataset...
Dataset copied: /content/drive/MyDrive/capstone_repo/instruction_formatted_admin_dataset.csv
Copying adapter files...
Adapter copied: /content/drive/MyDrive/capstone_repo/gemma-instruction-adapter

Copy verification:
Dataset exists: True
Adapter files copied: 6
  - gemma-instruction-adapter/README.md
  - gemma-instruction-adapter/adapter_model.safetensors
  - gemma-instruction-adapter/adapter_config.json
  - gemma-instruction-adapter/chat_template.jinja
  - gemma-instruction-adapter/tokenizer_config.json
  - gemma-instruction-adapter/tokenizer.json

Files successfully staged in the local repository!


In [46]:
%cd {CLONE_DIR}

# Stage all changes
!git add .

# Commit changes
!git commit -m "Add fine-tuned Gemma-2-2B instruction adapter and formatted dataset"

# Push to GitHub
print("Pushing changes to GitHub...")
!git push origin main


/content/capstone_repo
[main 93a5cab] Add fine-tuned Gemma-2-2B instruction adapter and formatted dataset
 7 files changed, 2584976 insertions(+)
 create mode 100644 gemma-instruction-adapter/README.md
 create mode 100644 gemma-instruction-adapter/adapter_config.json
 create mode 100644 gemma-instruction-adapter/adapter_model.safetensors
 create mode 100644 gemma-instruction-adapter/chat_template.jinja
 create mode 100644 gemma-instruction-adapter/tokenizer.json
 create mode 100644 gemma-instruction-adapter/tokenizer_config.json
 create mode 100644 instruction_formatted_admin_dataset.csv
Pushing changes to GitHub...
fatal: could not read Username for 'https://github.com': No such device or address


Add FastAPI UI to run the model

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio peft transformers torch
!pip install -q --upgrade torchao
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from google.colab import drive; drive.mount('/content/drive'); import uvicorn, torch; from fastapi import FastAPI; from fastapi.middleware.cors import CORSMiddleware; from pydantic import BaseModel; from pyngrok import ngrok; from peft import PeftModel; from transformers import AutoModelForCausalLM, AutoTokenizer; from huggingface_hub import notebook_login
notebook_login()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
base_model_id = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.bfloat16, device_map="cpu", low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/gemma-finetuned")
class PromptRequest(BaseModel): prompt: str
@app.post("/generate")
def generate_text(data: PromptRequest): inputs = tokenizer(data.prompt, return_tensors="pt"); outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7); response_text = tokenizer.decode(outputs[0], skip_special_tokens=True); return {"response": response_text}
public_url = ngrok.connect(8000)
print(f"🚀 YOUR LIVE API URL IS: {public_url.public_url}/generate")
await uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000)).serve()


**Run** model from HuggingFace